# 📡 TP Télécom 3GPP — Phase 3 : RAG Avancé

**Objectif** : Améliorer le pipeline RAG basique (Phase 2) avec des techniques avancées :
- **Chunking optimisé** — découpage intelligent des documents
- **Reranking** — reclassement des résultats avec un Cross-Encoder
- **BM25 + FAISS** — recherche hybride (mots-clés + sémantique)

```
Phase 1 ✅ → Phase 2 ✅ → [Phase 3] → Phase 4 → Phase 5 → Phase 6 → Phase 7
```

## 1. Installation des dépendances

In [ ]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu
!pip install -q transformers torch evaluate rouge_score
!pip install -q pandas matplotlib
print('✅ Installation terminée')

## 2. Imports & Configuration

In [ ]:
import json, time, pickle
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from transformers import pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')
print('✅ Imports effectués')

## 3. Chargement du Modèle et du Dataset

In [ ]:
# Chargement config Phase 1
SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
with open(f'{SAVE_DIR}\\pipeline_config.json') as f:
    config = json.load(f)
BEST_MODEL_ID = config['best_model_id']
print(f'✅ Modèle : {config["best_model_name"]}')

# Chargement LLM
print('🔄 Chargement du LLM...')
llm_pipeline = pipeline('text-generation', model=BEST_MODEL_ID, pad_token_id=50256)
print('✅ LLM chargé')

# Chargement dataset
TRAIN_PATH = r'C:\Users\HP\Downloads\TeleQnA_training.txt'
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

qa_pairs = []
for i, (key, item) in enumerate(train_data.items()):
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    qa_pairs.append({
        'id'              : i + 1,
        'question'        : item['question'],
        'reference_answer': str(answer_text) + '. ' + item.get('explanation', ''),
        'context'         : item.get('category', '3GPP')
    })

df_dataset = pd.DataFrame(qa_pairs[:10])
print(f'✅ Dataset chargé : {len(qa_pairs)} questions | 10 utilisées')

## 4. Construction du Corpus avec Chunking Optimisé

In [ ]:
def chunk_text(text, chunk_size=50, overlap=10):
    """Découpe un texte en chunks avec chevauchement."""
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Construction du corpus avec chunking
corpus = []
for key, item in train_data.items():
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    full_text   = (
        f"Question: {item['question']} "
        f"Answer: {answer_text}. "
        f"{item.get('explanation', '')}"
    )
    chunks = chunk_text(full_text)
    for chunk in chunks:
        corpus.append({
            'text'    : chunk,
            'category': item.get('category', '3GPP')
        })

print(f'✅ Corpus avec chunking : {len(corpus)} chunks')
print(f'   Exemple : {corpus[0]["text"][:150]}...')

## 5. Index FAISS + Index BM25

In [ ]:
# Modèle d'embedding
print('🔄 Chargement embedding model...')
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Embedding model chargé')

# Index FAISS (dense)
print('🔄 Création index FAISS...')
texts      = [doc['text'] for doc in corpus]
embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
faiss.normalize_L2(embeddings)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f'✅ Index FAISS : {index.ntotal} vecteurs')

# Index BM25 (sparse)
print('🔄 Création index BM25...')
tokenized = [doc['text'].lower().split() for doc in corpus]
bm25      = BM25Okapi(tokenized)
print('✅ Index BM25 créé')

## 6. Technique 1 — Recherche Hybride (BM25 + FAISS)

In [ ]:
def hybrid_retrieve(query, top_k=5, alpha=0.5):
    """
    Recherche hybride : combine BM25 (sparse) + FAISS (dense).
    alpha=0.5 → poids égal entre les deux méthodes.
    """
    # Scores BM25
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    bm25_norm   = (bm25_scores - bm25_min) / (bm25_max - bm25_min + 1e-9)

    # Scores FAISS
    q_emb = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(q_emb)
    faiss_scores = np.zeros(len(corpus))
    scores, indices = index.search(q_emb, min(top_k * 3, len(corpus)))
    for score, idx in zip(scores[0], indices[0]):
        faiss_scores[idx] = float(score)
    faiss_min, faiss_max = faiss_scores.min(), faiss_scores.max()
    faiss_norm = (faiss_scores - faiss_min) / (faiss_max - faiss_min + 1e-9)

    # Score hybride
    hybrid = alpha * bm25_norm + (1 - alpha) * faiss_norm
    top_idx = np.argsort(hybrid)[::-1][:top_k]

    return [{
        'text'         : corpus[i]['text'],
        'category'     : corpus[i]['category'],
        'hybrid_score' : float(hybrid[i]),
        'bm25_score'   : float(bm25_norm[i]),
        'dense_score'  : float(faiss_norm[i])
    } for i in top_idx]

# Test
test_q  = df_dataset.iloc[0]['question']
results = hybrid_retrieve(test_q, top_k=3)
print(f'🔍 Test Hybrid Retrieval :')
for i, r in enumerate(results):
    print(f'  [{i+1}] Hybride:{r["hybrid_score"]:.3f} | BM25:{r["bm25_score"]:.3f} | Dense:{r["dense_score"]:.3f}')
    print(f'       {r["text"][:100]}...')
print('✅ Hybrid Retrieval fonctionne !')

## 7. Technique 2 — Reranking avec Cross-Encoder

In [ ]:
print('🔄 Chargement Cross-Encoder...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('✅ Cross-Encoder chargé')

def retrieve_and_rerank(query, top_k_retrieve=10, top_k_final=3):
    """
    1. Récupère top_k_retrieve candidats via hybrid retrieval
    2. Reranking avec Cross-Encoder
    3. Retourne top_k_final résultats
    """
    # Étape 1 : Retrieval large
    candidates = hybrid_retrieve(query, top_k=top_k_retrieve)

    # Étape 2 : Reranking
    pairs         = [(query, c['text']) for c in candidates]
    rerank_scores = cross_encoder.predict(pairs)

    # Étape 3 : Tri final
    ranked = sorted(zip(rerank_scores, candidates), key=lambda x: x[0], reverse=True)
    return [{
        'text'         : c['text'],
        'category'     : c['category'],
        'rerank_score' : float(s),
        'hybrid_score' : c['hybrid_score']
    } for s, c in ranked[:top_k_final]]

# Test
reranked = retrieve_and_rerank(test_q)
print(f'\n🔍 Test Reranking :')
for i, r in enumerate(reranked):
    print(f'  [{i+1}] Rerank:{r["rerank_score"]:.3f} | {r["text"][:100]}...')
print('✅ Reranking fonctionne !')

## 8. Pipeline RAG Avancé Complet

In [ ]:
def advanced_rag_answer(question, max_new_tokens=100):
    """Pipeline RAG Avancé : hybrid retrieve → rerank → generate."""
    t0 = time.time()

    # Retrieve + Rerank
    docs    = retrieve_and_rerank(question, top_k_retrieve=10, top_k_final=3)
    context = ' | '.join([doc['text'][:150] for doc in docs])

    # Generate
    prompt = f'3GPP Context: {context} Question: {question} Answer:'
    result = llm_pipeline(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=50256,
        truncation=True
    )
    elapsed   = time.time() - t0
    full_text = result[0]['generated_text']
    answer    = full_text.replace(prompt, '').strip()
    return answer, docs, elapsed

def basic_rag_answer(question, max_new_tokens=100):
    """Pipeline RAG Basique (Phase 2) pour comparaison."""
    t0     = time.time()
    q_emb  = embed_model.encode([question]).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, 3)
    docs    = [corpus[i] for i in indices[0]]
    context = ' | '.join([doc['text'][:150] for doc in docs])
    prompt  = f'3GPP Context: {context} Question: {question} Answer:'
    result  = llm_pipeline(
        prompt, max_new_tokens=max_new_tokens,
        do_sample=False, pad_token_id=50256, truncation=True
    )
    elapsed   = time.time() - t0
    full_text = result[0]['generated_text']
    answer    = full_text.replace(prompt, '').strip()
    return answer, elapsed

print('✅ Pipelines RAG Basique et Avancé prêts')

## 9. Inférence : RAG Basique vs RAG Avancé

In [ ]:
results = []

for _, row in df_dataset.iterrows():
    print(f'\n🔍 Q{row["id"]} : {row["question"][:60]}...')

    # RAG Basique
    ans_basic, t_basic = basic_rag_answer(row['question'])
    print(f'   RAG Basique  ({t_basic:.1f}s) : {ans_basic[:80]}...')

    # RAG Avancé
    ans_adv, docs, t_adv = advanced_rag_answer(row['question'])
    print(f'   RAG Avancé   ({t_adv:.1f}s) : {ans_adv[:80]}...')

    results.append({
        'question_id' : row['id'],
        'question'    : row['question'],
        'reference'   : row['reference_answer'],
        'rag_basic'   : ans_basic,
        'rag_advanced': ans_adv,
        'time_basic'  : round(t_basic, 2),
        'time_advanced': round(t_adv, 2)
    })

df_results = pd.DataFrame(results)
SAVE_DIR   = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
df_results.to_csv(f'{SAVE_DIR}\\phase3_results.csv', index=False)
print(f'\n✅ Terminé — {len(df_results)} résultats sauvegardés')

## 10. Évaluation ROUGE — RAG Basique vs RAG Avancé

In [ ]:
from evaluate import load as load_metric
rouge = load_metric('rouge')

refs         = df_results['reference'].tolist()
scores_basic = rouge.compute(predictions=df_results['rag_basic'].tolist(),    references=refs)
scores_adv   = rouge.compute(predictions=df_results['rag_advanced'].tolist(), references=refs)

df_compare = pd.DataFrame([
    {
        'Approche'      : 'RAG Basique',
        'ROUGE-1'       : round(scores_basic['rouge1'], 4),
        'ROUGE-2'       : round(scores_basic['rouge2'], 4),
        'ROUGE-L'       : round(scores_basic['rougeL'], 4),
        'Temps moy. (s)': round(df_results['time_basic'].mean(), 2)
    },
    {
        'Approche'      : 'RAG Avancé',
        'ROUGE-1'       : round(scores_adv['rouge1'], 4),
        'ROUGE-2'       : round(scores_adv['rouge2'], 4),
        'ROUGE-L'       : round(scores_adv['rougeL'], 4),
        'Temps moy. (s)': round(df_results['time_advanced'].mean(), 2)
    }
])

print('📊 RAG Basique vs RAG Avancé :')
print(df_compare.to_string(index=False))
for m in ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']:
    delta = df_compare.iloc[1][m] - df_compare.iloc[0][m]
    print(f'  {m}: {delta:+.4f} avec RAG Avancé')

SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
df_compare.to_csv(f'{SAVE_DIR}\\phase3_evaluation.csv', index=False)
print('\n💾 Sauvegardé → phase3_evaluation.csv')

## 11. Visualisation — RAG Basique vs RAG Avancé

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('RAG Basique vs RAG Avancé — Dataset 3GPP', fontsize=15, fontweight='bold')
palette = ['#FF7043', '#7B1FA2']

# --- ROUGE scores ---
ax1     = axes[0]
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
x, w    = range(len(metrics)), 0.35
for i, (_, row) in enumerate(df_compare.iterrows()):
    bars = ax1.bar([p + i*w for p in x], [row[m] for m in metrics],
                   width=w, label=row['Approche'], color=palette[i], alpha=0.85)
    for b in bars:
        ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
                 f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.set_xticks([p+w/2 for p in x])
ax1.set_xticklabels(metrics)
ax1.set_ylim(0, 0.1)
ax1.set_title('Scores ROUGE', fontweight='bold')
ax1.set_ylabel('Score')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# --- Temps ---
ax2  = axes[1]
bars = ax2.bar(df_compare['Approche'], df_compare['Temps moy. (s)'], color=palette, alpha=0.85)
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
             f'{b.get_height():.1f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title("Temps moyen d'inférence", fontweight='bold')
ax2.set_ylabel('Secondes')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
plt.savefig(f'{SAVE_DIR}\\phase3_rag_basic_vs_advanced.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique → phase3_rag_basic_vs_advanced.png')

---
## ✅ Phase 3 Terminée

**Techniques utilisées :**
- ✅ Chunking optimisé avec chevauchement
- ✅ Recherche hybride BM25 + FAISS
- ✅ Reranking avec Cross-Encoder

**Fichiers produits :**
- `phase3_results.csv`
- `phase3_evaluation.csv`
- `phase3_rag_basic_vs_advanced.png`

**➡️ Prochaine étape : Phase 4 — Fine-Tuning avec QLoRA**